# Smoke Test MinIO with Polars

Notebook này thực hiện smoke test kết nối đến MinIO Object Storage bằng cách:
1. Khởi tạo Polars DataFrame mẫu.
2. Ghi DataFrame thành file Parquet lên MinIO bucket `lakehouse` sử dụng `s3fs`.
3. Đọc ngược lại file từ MinIO và kiểm tra tính toàn vẹn của dữ liệu.

In [2]:
import polars as pl
import s3fs
import os
from dotenv import load_dotenv

# Load environment variables từ file .env nếu có
load_dotenv(dotenv_path="../.env")

True

## 1. Khởi tạo thông tin kết nối MinIO

In [2]:
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://localhost:9000")
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID", "admin")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "admin123")

# Khởi tạo S3 FileSystem để kết nối với MinIO
fs = s3fs.S3FileSystem(
    key=AWS_ACCESS_KEY_ID,
    secret=AWS_SECRET_ACCESS_KEY,
    client_kwargs={"endpoint_url": MINIO_ENDPOINT}
)
print("S3FileSystem initialized successfully!")

S3FileSystem initialized successfully!


## 2. Tạo một Polars DataFrame mẫu

In [3]:
df = pl.DataFrame({
    "symbol": ["AAA", "BBB", "CCC"],
    "price": [15.4, 20.1, 10.5],
    "volume": [100000, 250000, 50000],
    "trading_date": ["2026-06-12", "2026-06-12", "2026-06-12"]
})
print("Sample DataFrame:")
print(df)

Sample DataFrame:
shape: (3, 4)
┌────────┬───────┬────────┬──────────────┐
│ symbol ┆ price ┆ volume ┆ trading_date │
│ ---    ┆ ---   ┆ ---    ┆ ---          │
│ str    ┆ f64   ┆ i64    ┆ str          │
╞════════╪═══════╪════════╪══════════════╡
│ AAA    ┆ 15.4  ┆ 100000 ┆ 2026-06-12   │
│ BBB    ┆ 20.1  ┆ 250000 ┆ 2026-06-12   │
│ CCC    ┆ 10.5  ┆ 50000  ┆ 2026-06-12   │
└────────┴───────┴────────┴──────────────┘


## 3. Ghi Polars DataFrame thành file Parquet lên MinIO

In [4]:
s3_target_path = "s3://lakehouse/staging/smoke_test_df.parquet"

print(f"Writing DataFrame to {s3_target_path}...")
with fs.open(s3_target_path, "wb") as f:
    df.write_parquet(f)
print("Parquet file written to MinIO successfully!")

Writing DataFrame to s3://lakehouse/staging/smoke_test_df.parquet...
Parquet file written to MinIO successfully!


## 4. Đọc ngược lại file từ MinIO để xác nhận kết nối

In [5]:
print(f"Reading DataFrame back from {s3_target_path}...")
with fs.open(s3_target_path, "rb") as f:
    df_read = pl.read_parquet(f)

print("Read back DataFrame:")
print(df_read)

# Kiểm tra tính toàn vẹn của dữ liệu
assert df.equals(df_read), "Error: Data integrity mismatch!"
print("SUCCESS: Data integrity check passed!")

Reading DataFrame back from s3://lakehouse/staging/smoke_test_df.parquet...
Read back DataFrame:
shape: (3, 4)
┌────────┬───────┬────────┬──────────────┐
│ symbol ┆ price ┆ volume ┆ trading_date │
│ ---    ┆ ---   ┆ ---    ┆ ---          │
│ str    ┆ f64   ┆ i64    ┆ str          │
╞════════╪═══════╪════════╪══════════════╡
│ AAA    ┆ 15.4  ┆ 100000 ┆ 2026-06-12   │
│ BBB    ┆ 20.1  ┆ 250000 ┆ 2026-06-12   │
│ CCC    ┆ 10.5  ┆ 50000  ┆ 2026-06-12   │
└────────┴───────┴────────┴──────────────┘
SUCCESS: Data integrity check passed!
